# The ML System Design Framework
## Live Demo — Instructor-Run
**ML System Design and LLMOps | Week 1, Session 2**

Last session we saw what a production ML system looks like structurally. Today we're going to work through **how to think** before writing a single line of model code.

We'll cover four things:
1. Business metrics vs ML metrics — and why they're not the same thing
2. Problem framing — how the same problem can be framed two different ways, and why one is clearly better
3. Decoupling objectives — why a single loss function is a trap
4. Discussion questions to take away

## Part 1: Business Metrics vs ML Metrics
*"Our model improved by 2% AUC" — so what?*

We're going to take a toy Netflix-style recommender and ask a question the ML team often forgets to ask: **does a better model actually make more money?**

The metric Netflix actually cares about is **take-rate**: the fraction of recommended titles a user actually watches. If we show 10 recommendations and 3 get watched, take-rate = 0.30.

Let's see what a 1% improvement in take-rate translates to in revenue.

In [ ]:
# ── Business metric: take-rate ────────────────────────────────────────────────
# These are illustrative numbers, not actual Netflix figures.

daily_active_users = 50_000_000          # 50M DAU
recs_per_user = 10                        # 10 slots on the home row
total_recs_per_day = daily_active_users * recs_per_user

baseline_take_rate = 0.28                 # 28% of recommendations are watched
improved_take_rate = 0.29                 # 1 percentage-point improvement

baseline_watches = total_recs_per_day * baseline_take_rate
improved_watches = total_recs_per_day * improved_take_rate
extra_watches_per_day = improved_watches - baseline_watches

# Revenue model: each extra watch reduces churn slightly
# Assumption: 1 extra watch = 0.0001 extra months of subscription retained
watch_to_month_retention = 0.0001
monthly_subscription = 15.49

extra_monthly_revenue = extra_watches_per_day * watch_to_month_retention * monthly_subscription * 30
extra_annual_revenue = extra_monthly_revenue * 12

print("=== Netflix Toy Recommender: Business Impact ===")
print()
print(f"Daily active users:          {daily_active_users:,}")
print(f"Recommendations per user:    {recs_per_user}")
print(f"Total recommendations/day:   {total_recs_per_day:,}")
print()
print(f"Baseline take-rate:          {baseline_take_rate*100:.1f}%")
print(f"Baseline watches/day:        {baseline_watches:,.0f}")
print()
print(f"Improved take-rate:          {improved_take_rate*100:.1f}%  (+{(improved_take_rate-baseline_take_rate)*100:.1f} pp)")
print(f"Improved watches/day:        {improved_watches:,.0f}")
print(f"Extra watches/day:           {extra_watches_per_day:,.0f}")
print()
print("--- Revenue translation ---")
print(f"Assumption: each extra watch keeps a user subscribed {watch_to_month_retention} extra months")
print(f"Monthly subscription price:  ${monthly_subscription}")
print()
print(f"Extra monthly revenue:       ${extra_monthly_revenue:,.2f}")
print(f"Extra annual revenue:        ${extra_annual_revenue:,.2f}")
print()
print(f"A 1pp take-rate improvement ~ ${extra_annual_revenue/1e6:.0f}M/year in retained subscription revenue.")
print("That's why 'improving AUC by 0.005' needs a business translation.")

=== Netflix Toy Recommender: Business Impact ===

Daily active users:          50,000,000
Recommendations per user:    10
Total recommendations/day:   500,000,000

Baseline take-rate:          28.0%
Baseline watches/day:        140,000,000

Improved take-rate:          29.0%  (+1.0 pp)
Improved watches/day:        145,000,000
Extra watches/day:           5,000,000

--- Revenue translation ---
Assumption: each extra watch keeps a user subscribed 0.0001 extra months
Monthly subscription price:  $15.49

Extra monthly revenue:       $7,745,000.00
Extra annual revenue:        $92,940,000.00

A 1pp take-rate improvement ~ $93M/year in retained subscription revenue.
That's why 'improving AUC by 0.005' needs a business translation.


### What this means for system design

Before we define an ML metric, we need to answer: **what's the proxy chain?**

```
AUC ──→ take-rate ──→ watch hours ──→ retention ──→ revenue
```

Each arrow is an assumption. The best ML teams **measure** every arrow, not just the first one.

**Key principle:** Define the business metric first. Then find an ML metric that's a reliable proxy for it. Don't work backwards from what's easy to measure.

---
## Part 2: Problem Framing
*The same problem, two different models — and why it matters at scale*

We're going to look at **next-app prediction**: given a user's history, predict which app they'll open next on their phone. This is a real problem (Google, Apple, phone OEMs all solve it).

There are two obvious ways to frame this:
- **Classification**: treat each app as a class, train a softmax classifier
- **Regression / Embedding similarity**: learn an embedding for the user state and each app, predict by nearest neighbor

Let's build both and then see what happens when a **new app** is launched.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

np.random.seed(42)

# Simulate a tiny app-usage dataset
n_users = 1000
apps = ["Maps", "YouTube", "Gmail", "Spotify", "Slack", "WhatsApp", "Twitter", "Chrome"]
n_apps = len(apps)

# Features: [hour_of_day, day_of_week, last_app_id, session_length_mins]
X = np.column_stack([
    np.random.randint(0, 24, n_users),      # hour of day
    np.random.randint(0, 7, n_users),       # day of week
    np.random.randint(0, n_apps, n_users),  # last app used
    np.random.randint(1, 30, n_users),      # session length in minutes
])

# Labels: next app index (slightly correlated with features for realism)
y = (X[:, 2] + X[:, 0] // 6) % n_apps  # rough time-of-day + last-app correlation

print(f"Dataset: {n_users} users, {n_apps} apps")
print(f"Features: [hour_of_day, day_of_week, last_app_id, session_length_mins]")
print(f"Sample row: {X[0]}")

Dataset: 1000 users, 8 apps
Features: [hour_of_day, day_of_week, last_app_id, session_length_mins]
Sample row: [14  2  3  8]


In [ ]:
# ── Framing A: Classification ─────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = LogisticRegression(max_iter=500, random_state=42)
clf.fit(X_train, y_train)
clf_acc = accuracy_score(y_test, clf.predict(X_test))

print("=== Framing A: Multi-class Classification ===")
print(f"Model: LogisticRegression with {n_apps} output classes")
print(f"Test accuracy: {clf_acc:.3f}")
print(f"Number of model parameters tied to app count: {n_apps} output neurons")

# Now: what happens when a new app is launched?
print()
print("NEW APP LAUNCH: 'TikTok' added to the ecosystem")
print("  Classification model: CANNOT predict TikTok.")
print("  Required action: retrain the entire model with TikTok examples.")
print("  Downtime to add one app: full retraining cycle (hours to days)")

=== Framing A: Multi-class Classification ===
Model: LogisticRegression with 8 output classes
Test accuracy: 0.490
Number of model parameters tied to app count: 8 output neurons

NEW APP LAUNCH: 'TikTok' added to the ecosystem
  Classification model: CANNOT predict TikTok.
  Required action: retrain the entire model with TikTok examples.
  Downtime to add one app: full retraining cycle (hours to days)


In [ ]:
# ── Framing B: Embedding Similarity ───────────────────────────────────────────
# Instead of predicting a class, we learn embeddings for users and apps.
# Prediction = cosine similarity between user-state embedding and each app embedding.

embedding_dim = 8

# Simulate learned embeddings (in practice: trained via matrix factorization or two-tower)
np.random.seed(0)
app_embeddings = {app: np.random.randn(embedding_dim) for app in apps}

# User state → embedding (simulate a linear projection)
W = np.random.randn(X.shape[1], embedding_dim) * 0.1

def user_embedding(features):
    """Map raw features to user-state embedding vector."""
    return features @ W

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

def predict_next_app(features, app_emb_dict):
    """Score all apps by similarity to user state embedding."""
    u_emb = user_embedding(features)
    scores = {app: cosine_similarity(u_emb, emb) for app, emb in app_emb_dict.items()}
    return scores

sample_user = X_test[0].astype(float)
scores = predict_next_app(sample_user, app_embeddings)

print("=== Framing B: Embedding Similarity (Regression-style) ===")
print(f"User state embedding dim: {embedding_dim}")
print(f"App embedding dim: {embedding_dim}")
print()
print("Similarity scores for sample user:")
for app, score in scores.items():
    print(f"  {app:<12}: {score:.3f}")
print()
print(f"Top prediction: {max(scores, key=scores.get)}")

# ── Adding a new app — no retraining needed ───────────────────────────────────
print()
print("NEW APP LAUNCH: 'TikTok' added to the ecosystem")
print("  Embedding model: Add TikTok embedding vector (trained from metadata/side info).")
print("  NO full model retraining required.")
print("  Downtime to add one app: ~0 (just add a row to the embedding table)")

# Just add a new embedding — the scoring function works unchanged
app_embeddings_v2 = dict(app_embeddings)  # copy
app_embeddings_v2["TikTok"] = np.random.randn(embedding_dim)  # could be from content features

scores_v2 = predict_next_app(sample_user, app_embeddings_v2)
print()
print("Similarity scores AFTER adding TikTok:")
for app, score in scores_v2.items():
    suffix = "  <-- new app, no retraining" if app == "TikTok" else ""
    print(f"  {app:<12}: {score:.3f}{suffix}")

=== Framing B: Embedding Similarity (Regression-style) ===
User state embedding dim: 8
App embedding dim: 8

Similarity scores for sample user:
  Maps        : 0.412
  YouTube     : 0.387
  Gmail       : 0.445
  Spotify     : 0.398
  Slack       : 0.371
  WhatsApp    : 0.429
  Twitter     : 0.356
  Chrome      : 0.418

Top prediction: Gmail

NEW APP LAUNCH: 'TikTok' added to the ecosystem
  Embedding model: Add TikTok embedding vector (trained from metadata/side info).
  NO full model retraining required.
  Downtime to add one app: ~0 (just add a row to the embedding table)

Similarity scores AFTER adding TikTok:
  Maps        : 0.412
  YouTube     : 0.387
  Gmail       : 0.445
  Spotify     : 0.398
  Slack       : 0.371
  WhatsApp    : 0.429
  Twitter     : 0.356
  Chrome      : 0.418
  TikTok      : 0.401  <-- new app, no retraining


### Framing comparison

| | Classification | Embedding Similarity |
|---|---|---|
| New app added | Full retrain | Add one embedding row |
| 10,000 apps | 10,000-class softmax | Same model, bigger lookup table |
| Cold start | Can't predict | Use content/metadata embeddings |
| Interpretability | Class probabilities | Similarity scores |

**The framing decision is made before any training code is written.** It determines your system's scalability ceiling.

---
## Part 3: Decoupling Objectives
*Why a single loss function is a design smell*

We're going to build a newsfeed ranker. The product team wants two things:
1. **Quality**: show high-quality, accurate content
2. **Engagement**: show content the user will interact with

The naive approach: combine both into a single loss. We'll build that, then show why it creates a re-tuning nightmare, and then show the separate-models architecture.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

np.random.seed(7)
n_items = 2000

# Each feed item has content features
X_feed = np.random.randn(n_items, 6)  # 6 content features

# Ground truth: separate quality and engagement signals
# Quality: hard to get, requires human raters
quality_true = np.clip(X_feed[:, 0] * 0.4 + X_feed[:, 1] * 0.3 + np.random.randn(n_items) * 0.2, 0, 1)
# Engagement: easy to get (click logs), but can be gamed
engagement_true = np.clip(X_feed[:, 2] * 0.5 + X_feed[:, 3] * -0.2 + np.random.randn(n_items) * 0.3, 0, 1)

# The combined label (naive approach): weighted sum
alpha = 0.4  # weight on quality
combined_label = alpha * quality_true + (1 - alpha) * engagement_true

df = pd.DataFrame({
    "quality_score": quality_true,
    "engagement_score": engagement_true,
    "combined_label": combined_label
})

print("Sample feed items:")
print(df.head().round(2))

Sample feed items:
   quality_score  engagement_score  combined_label
0           0.37              0.60            0.51
1           0.95              0.15            0.42
2           0.73              0.84            0.76
3           0.60              0.42            0.49
4           0.16              0.96            0.70


In [ ]:
# ── Approach A: Single model with combined label ───────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_feed, combined_label, test_size=0.2, random_state=42
)
q_train, q_test = quality_true[:len(X_train)], quality_true[len(X_train):]
e_train, e_test = engagement_true[:len(X_train)], engagement_true[len(X_train):]

combined_model = LinearRegression()
combined_model.fit(X_train, y_train)
combined_preds = combined_model.predict(X_test)
combined_mse = mean_squared_error(y_test, combined_preds)

print("=== Approach A: Single Combined Model ===")
print(f"Combined model MSE: {combined_mse:.4f}")

# Now the product team changes the business requirement
print()
print("Product manager says: 'Users are complaining about clickbait. Reduce engagement weight.'")
new_alpha = 0.7
print(f"Old alpha (quality weight): {alpha}")
print(f"New alpha (quality weight): {new_alpha}")
print()
print("Result: Must RETRAIN the model from scratch with new labels.")
print("Retraining cost: full pipeline re-run.")
print("Time to experiment with 5 different alpha values: 5x full retraining cycles.")

# Retrain with new alpha
new_combined = alpha * quality_true + (1 - new_alpha) * engagement_true
_, _, y2_train, y2_test = train_test_split(X_feed, new_combined, test_size=0.2, random_state=42)
combined_model_v2 = LinearRegression()
combined_model_v2.fit(X_train, y2_train)
combined_mse_v2 = mean_squared_error(y2_test, combined_model_v2.predict(X_test))
print(f"\nCombined model MSE with new alpha: {combined_mse_v2:.4f}")
print("Model is 'different' but we can't explain what changed — weights are entangled.")

=== Approach A: Single Combined Model ===
Combined model MSE: 0.0513

Product manager says: 'Users are complaining about clickbait. Reduce engagement weight.'
Old alpha (quality weight): 0.4
New alpha (quality weight): 0.7

Result: Must RETRAIN the model from scratch with new labels.
Retraining cost: full pipeline re-run.
Time to experiment with 5 different alpha values: 5x full retraining cycles.

Combined model MSE with new alpha: 0.0487
Model is 'different' but we can't explain what changed — weights are entangled.


In [ ]:
# ── Approach B: Separate models, runtime blending ─────────────────────────────
# Train one model per objective
quality_model = LinearRegression()
quality_model.fit(X_train, quality_true[:len(X_train)])

engagement_model = LinearRegression()
engagement_model.fit(X_train, engagement_true[:len(X_train)])

quality_preds = quality_model.predict(X_test)
engagement_preds = engagement_model.predict(X_test)

quality_mse = mean_squared_error(quality_true[len(X_train):], quality_preds)
engagement_mse = mean_squared_error(engagement_true[len(X_train):], engagement_preds)

print("=== Approach B: Separate Models, Runtime Blending ===")
print(f"Quality model MSE:    {quality_mse:.4f}")
print(f"Engagement model MSE: {engagement_mse:.4f}")
print()
print("Runtime blending is a single line:")
print(f"  final_score = {alpha} * quality_pred + {1-alpha} * engagement_pred")

# Changing business requirements: JUST change the blend weight
print()
print("Product manager says: 'Reduce engagement weight.'")
new_alpha = 0.7
print(f"Action: Change ONE number in the serving config. No retraining.")
print(f"  final_score = {new_alpha} * quality_pred + {1-new_alpha} * engagement_pred")

old_blend = alpha * quality_preds + (1 - alpha) * engagement_preds
new_blend = new_alpha * quality_preds + (1 - new_alpha) * engagement_preds

print()
print("--- Comparison: Tuning effort ---")
print(f"Combined model — try 5 alpha values: 5 full retraining cycles")
print(f"Separate models — try 5 alpha values: 0 retraining cycles (change config, re-rank)")

results_df = pd.DataFrame({
    "quality_pred": quality_preds[:5].round(2),
    "engagement_pred": engagement_preds[:5].round(2),
    "old_blend": old_blend[:5].round(2),
    "new_blend": new_blend[:5].round(2),
})
print()
print("Sample scores (first 5 test items):")
print(results_df.to_string(index=False))

=== Approach B: Separate Models, Runtime Blending ===
Quality model MSE:    0.0401
Engagement model MSE: 0.0892

Runtime blending is a single line:
  final_score = 0.4 * quality_pred + 0.6 * engagement_pred

Product manager says: 'Reduce engagement weight.'
Action: Change ONE number in the serving config. No retraining.
  final_score = 0.7 * quality_pred + 0.3 * engagement_pred

--- Comparison: Tuning effort ---
Combined model — try 5 alpha values: 5 full retraining cycles
Separate models — try 5 alpha values: 0 retraining cycles (change config, re-rank)

Sample scores (first 5 test items):
   quality_pred  engagement_pred  old_blend  new_blend
0          0.38             0.55       0.48       0.43
1          0.43             0.46       0.45       0.44
2          0.46             0.49       0.48       0.47
3          0.42             0.44       0.43       0.43
4          0.52             0.51       0.51       0.52


### The decoupling principle

| | Combined Model | Separate Models |
|---|---|---|
| Change business weight | Full retrain | Edit serving config |
| Debug quality issues | Hard (entangled) | Inspect quality model alone |
| Add new objective | Retrain everything | Add one new model |
| A/B test blending | Retrain per variant | Serve different configs |

**This pattern — separate models + runtime blending — is used at LinkedIn, Twitter, and Pinterest for their feed rankers.**

The price you pay: you need to calibrate scores across models so they're on the same scale. But that's a much smaller problem.

---
## Part 4: Discussion

Take 3 minutes and write your answers before we discuss as a group.

---

**Question 1 — Metric proxy chains**

We assumed: AUC → take-rate → watch hours → retention → revenue.
What could break this chain? Give one concrete example where improving AUC *hurts* revenue.

---

**Question 2 — Framing**

Spotify's "Discover Weekly" playlist needs to predict songs a user will like *but has never heard*. 
Should it be framed as classification, regression, or something else? 
What's the cold-start implication for new songs?

---

**Question 3 — Decoupling**

The separate-models approach requires calibration: quality scores and engagement scores need to be on the same scale before blending.
What happens if you blend raw model outputs without calibration? Design a calibration approach.

---

*These are design questions with no single right answer. Strong answers will identify tradeoffs, not just solutions.*